In [5]:
import pathlib, sys
import numpy as np
import plotly.graph_objects as go
import torch
from PIL import Image

HERE = pathlib.Path.cwd()
sys.path[:0] = [str(HERE), str(HERE.parent / "mhr_posing")]

import losses, mhr_scene, model_spatial
from dataset import IMAGENET_MEAN, IMAGENET_STD
from mhr_kit.model import faces
from prepare import KEYPOINTS, PELVIS, ik

In [6]:
IMAGE = '/Users/rzt/Desktop/Screenshot 2026-09-23 at 10.21.26 AM.png'
W = 0.30

net = model_spatial.PoseNet(torch.load(HERE / "model_spatial.pt", map_location="cpu")["backbone"], pretrained=False)
state = torch.load(HERE / "model_spatial.pt", map_location="cpu")
net.load_state_dict(state["weights"])
net.eval()
body = losses.Body(ik.Rig())

picture = Image.open(IMAGE).convert("RGB")
side = max(picture.size)
square = Image.new("RGB", (side, side), (124, 116, 104))
square.paste(picture, ((side - picture.size[0]) // 2, (side - picture.size[1]) // 2))
rgb = np.asarray(square.resize((256, 256), Image.BILINEAR), np.float32) / 255.0
image = torch.from_numpy(((rgb - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1))[None]

context = torch.tensor([[0.0, 0.0, W]])
box = torch.tensor([[128.0, 128.0, 256.0]])
intrinsics = torch.tensor([[256.0 / W, 256.0 / W, 128.0, 128.0]])

with torch.no_grad():
    out = net(image, context)

local = body.joints(out["angles"], out["scales"])
translation = losses.translation_from(out["camera"], box, intrinsics)
view = losses.place_in_view(local, out["rotation"], translation).detach()

parameters = torch.zeros(1, losses.N_MHR_PARAMETERS)
parameters = parameters.index_copy(1, body.angles_at, out["angles"])
parameters = parameters.index_copy(1, body.scales_at, out["scales"])
with torch.no_grad():
    vertices, rig_state = body.rig.model(torch.zeros(1, losses.N_IDENTITY), parameters, None)
hips = rig_state[:, body.joints_at[PELVIS], :3]
mesh = losses.place_in_view(vertices - hips[:, None, :], out["rotation"], translation).detach().numpy()[0]

BONES = [("pelvis", "spine"), ("spine", "neck"), ("neck", "head"), ("head", "head_top"),
         ("neck", "left_shoulder"), ("left_shoulder", "left_elbow"), ("left_elbow", "left_wrist"),
         ("neck", "right_shoulder"), ("right_shoulder", "right_elbow"), ("right_elbow", "right_wrist"),
         ("pelvis", "left_hip"), ("left_hip", "left_knee"), ("left_knee", "left_ankle"),
         ("pelvis", "right_hip"), ("right_hip", "right_knee"), ("right_knee", "right_ankle")]
LINKS = [(KEYPOINTS.index(a), KEYPOINTS.index(b)) for a, b in BONES]

def stick(points):
    points = np.asarray(points)
    return [[v for a, b in LINKS for v in (points[a, i], points[b, i], None)]
            for i in range(points.shape[1])]

go.Figure(go.Image(z=rgb * 255.0)).update_layout(
    height=400, width=400, margin=dict(l=0, r=0, t=0, b=0),
    xaxis=dict(visible=False), yaxis=dict(visible=False)).show()

points = ((losses.to_crop_units(losses.project(view, intrinsics), box) + 1.0) / 2.0 * 256).numpy()[0]
x, y = stick(points)
go.Figure([go.Image(z=rgb * 255.0, hoverinfo="skip"),
           go.Scatter(x=x, y=y, mode="lines+markers", line=dict(color="orange", width=3),
                      marker=dict(size=5))]).update_layout(
    height=400, width=400, margin=dict(l=0, r=0, t=0, b=0), showlegend=False,
    xaxis=dict(visible=False, range=[0, 256]),
    yaxis=dict(visible=False, range=[256, 0], scaleanchor="x")).show()

centres, reach = mhr_scene._framing([[mesh]])
mx, my, mz = stick(mhr_scene.plotted(view.numpy()[0]))
go.Figure([mhr_scene._body(mesh, faces(body.rig.model), "lightsteelblue", "body"),
           go.Scatter3d(x=mx, y=my, z=mz, mode="lines+markers",
                        line=dict(color="orange", width=5), marker=dict(size=3))]).update_layout(
    height=640, margin=dict(l=0, r=0, t=0, b=0), showlegend=False,
    scene=mhr_scene._scene(centres[0], reach)).show()



In [7]:
IMAGE = '/Users/rzt/Desktop/Screenshot 2026-09-23 at 10.12.54 AM.png'
W = 0.30

net = model_spatial.PoseNet(torch.load(HERE / "model_spatial.pt", map_location="cpu")["backbone"], pretrained=False)
state = torch.load(HERE / "model_spatial.pt", map_location="cpu")
net.load_state_dict(state["weights"])
net.eval()
body = losses.Body(ik.Rig())

picture = Image.open(IMAGE).convert("RGB")
side = max(picture.size)
square = Image.new("RGB", (side, side), (124, 116, 104))
square.paste(picture, ((side - picture.size[0]) // 2, (side - picture.size[1]) // 2))
rgb = np.asarray(square.resize((256, 256), Image.BILINEAR), np.float32) / 255.0
image = torch.from_numpy(((rgb - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1))[None]

context = torch.tensor([[0.0, 0.0, W]])
box = torch.tensor([[128.0, 128.0, 256.0]])
intrinsics = torch.tensor([[256.0 / W, 256.0 / W, 128.0, 128.0]])

with torch.no_grad():
    out = net(image, context)

local = body.joints(out["angles"], out["scales"])
translation = losses.translation_from(out["camera"], box, intrinsics)
view = losses.place_in_view(local, out["rotation"], translation).detach()

parameters = torch.zeros(1, losses.N_MHR_PARAMETERS)
parameters = parameters.index_copy(1, body.angles_at, out["angles"])
parameters = parameters.index_copy(1, body.scales_at, out["scales"])
with torch.no_grad():
    vertices, rig_state = body.rig.model(torch.zeros(1, losses.N_IDENTITY), parameters, None)
hips = rig_state[:, body.joints_at[PELVIS], :3]
mesh = losses.place_in_view(vertices - hips[:, None, :], out["rotation"], translation).detach().numpy()[0]

BONES = [("pelvis", "spine"), ("spine", "neck"), ("neck", "head"), ("head", "head_top"),
         ("neck", "left_shoulder"), ("left_shoulder", "left_elbow"), ("left_elbow", "left_wrist"),
         ("neck", "right_shoulder"), ("right_shoulder", "right_elbow"), ("right_elbow", "right_wrist"),
         ("pelvis", "left_hip"), ("left_hip", "left_knee"), ("left_knee", "left_ankle"),
         ("pelvis", "right_hip"), ("right_hip", "right_knee"), ("right_knee", "right_ankle")]
LINKS = [(KEYPOINTS.index(a), KEYPOINTS.index(b)) for a, b in BONES]

def stick(points):
    points = np.asarray(points)
    return [[v for a, b in LINKS for v in (points[a, i], points[b, i], None)]
            for i in range(points.shape[1])]

go.Figure(go.Image(z=rgb * 255.0)).update_layout(
    height=400, width=400, margin=dict(l=0, r=0, t=0, b=0),
    xaxis=dict(visible=False), yaxis=dict(visible=False)).show()

points = ((losses.to_crop_units(losses.project(view, intrinsics), box) + 1.0) / 2.0 * 256).numpy()[0]
x, y = stick(points)
go.Figure([go.Image(z=rgb * 255.0, hoverinfo="skip"),
           go.Scatter(x=x, y=y, mode="lines+markers", line=dict(color="orange", width=3),
                      marker=dict(size=5))]).update_layout(
    height=400, width=400, margin=dict(l=0, r=0, t=0, b=0), showlegend=False,
    xaxis=dict(visible=False, range=[0, 256]),
    yaxis=dict(visible=False, range=[256, 0], scaleanchor="x")).show()

centres, reach = mhr_scene._framing([[mesh]])
mx, my, mz = stick(mhr_scene.plotted(view.numpy()[0]))
go.Figure([mhr_scene._body(mesh, faces(body.rig.model), "lightsteelblue", "body"),
           go.Scatter3d(x=mx, y=my, z=mz, mode="lines+markers",
                        line=dict(color="orange", width=5), marker=dict(size=3))]).update_layout(
    height=640, margin=dict(l=0, r=0, t=0, b=0), showlegend=False,
    scene=mhr_scene._scene(centres[0], reach)).show()
